In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import pickle
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [3]:
DATA_DIR = Path("data")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(exist_ok=True, parents=True)

# Dataset paths
AUTSL_DIR = DATA_DIR / "AUTSL"
WLASL_DIR = DATA_DIR / "WLASL"
EMOSIGN_DIR = DATA_DIR / "emosign"

In [4]:

def extract_frames_from_video(video_path, num_frames=8, target_size=(64,64)):
    """
    Extract uniform number of frames from a video
    
    Args:
        video_path: Path to video file
        num_frames: Number of frames to extract
        target_size: (height, width) for resizing
    
    Returns:
        numpy array of shape [num_frames, height, width, 3]
    """
    cap = cv2.VideoCapture(str(video_path))
    
    if not cap.isOpened():
        print(f"Error opening video: {video_path}")
        return None
    
    # Get total frames in video
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        cap.release()
        return None
    
    # Calculate frame indices to sample
    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    
    frames = []
    
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        
        if ret:
            # Resize frame
            frame = cv2.resize(frame, target_size)
            # Convert BGR to RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        else:
            # If frame read fails, duplicate last frame
            if frames:
                frames.append(frames[-1])
            else:
                frames.append(np.zeros((*target_size, 3), dtype=np.uint8))
    
    cap.release()
    
    # Stack frames into array
    frames = np.stack(frames, axis=0)  # [num_frames, H, W, 3]
    
    return frames




In [5]:
def preprocess_autsl_dataset(split='train', num_frames=8, target_size=(64, 64)):
    """
    Preprocess all videos in a split of AUTSL dataset
    """
    
    # Load CSV
    csv_path = AUTSL_DIR / f"{split}.csv"
    df = pd.read_csv(csv_path)
    
    video_col = df.columns[0]  # 'name'
    label_col = df.columns[1]  # 'class_id'
    
    # Video directory
    video_dir = AUTSL_DIR / split
    
    # Storage for processed data
    data = {'frames': [], 'labels': [], 'video_names': []}
    
    print(f"Processing {split} split ({len(df)} videos)...")
    print(f"Looking for videos in: {video_dir}")
    
    found_count = 0
    not_found_count = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        video_name = row[video_col]
        label = row[label_col]
        
        # Try to find the video file
        video_path = video_dir / video_name
        
        # Also try without _color if not found
        if not video_path.exists():
            base_name = video_name.replace("_color.mp4", ".mp4")
            video_path = video_dir / base_name
        
        if not video_path.exists():
            not_found_count += 1
            if not_found_count <= 5:  # Print only first 5 missing
                print(f"Warning: Video not found: {video_name}")
            continue
        
        found_count += 1
        
        # Extract frames
        frames = extract_frames_from_video(video_path, num_frames, target_size)
        
        if frames is not None:
            data['frames'].append(frames)
            data['labels'].append(label)
            data['video_names'].append(video_name)
    
    print(f"\n--- AUTSL {split} Summary ---")
    print(f"Total in CSV: {len(df)}")
    print(f"Found: {found_count}")
    print(f"Not found: {not_found_count}")
    
    if len(data['frames']) == 0:
        print(f"ERROR: No videos found for {split} split!")
        return None
    
    # Convert to numpy arrays
    data['frames'] = np.array(data['frames'], dtype=np.uint8)
    data['labels'] = np.array(data['labels'])
    
    # Save to disk
    save_path = PROCESSED_DIR / f"autsl_{split}.pkl"
    with open(save_path, 'wb') as f:
        pickle.dump(data, f)
    
    print(f" Saved {len(data['frames'])} videos to {save_path}")
    print(f" Frames shape: {data['frames'].shape}")
    print(f" Labels shape: {data['labels'].shape}")
    
    return data


def process_all_autsl():
    """Process all AUTSL splits"""
    splits = ['train', 'val', 'test']
    
    for split in splits:
        print(f"\n{'='*60}")
        print(f"Processing AUTSL {split.upper()} split")
        print(f"{'='*60}")
        
        # Verify files exist before processing
        csv_path = AUTSL_DIR / f"{split}.csv"
        folder_path = AUTSL_DIR / split
        
        if not csv_path.exists():
            print(f"CSV not found: {csv_path}")
            continue
        
        if not folder_path.exists():
            print(f"Folder not found: {folder_path}")
            continue
        
        # Count videos in folder
        video_count = len(list(folder_path.glob("*.mp4")))
        print(f"Found {video_count} videos in folder")
        
        # Process
        result = preprocess_autsl_dataset(split=split, num_frames=8, target_size=(64, 64))
        
        if result is None:
            print(f"Failed to process {split} split")
        else:
            print(f"Successfully processed {split} split")




In [6]:
process_all_autsl()


Processing AUTSL TRAIN split
Found 28142 videos in folder
Processing train split (28142 videos)...
Looking for videos in: data\AUTSL\train


  9%|▉         | 2512/28142 [01:36<50:14,  8.50it/s]

Error opening video: data\AUTSL\train\signer2_sample1161_color.mp4


 10%|█         | 2862/28142 [01:49<17:28, 24.12it/s]

Error opening video: data\AUTSL\train\signer3_sample152_color.mp4


...


--- AUTSL val Summary ---
Total in CSV: 4418
Found: 4418
Not found: 0
 Saved 4418 videos to data\processed\autsl_val.pkl
 Frames shape: (4418, 8, 64, 64, 3)
 Labels shape: (4418,)
Successfully processed val split

Processing AUTSL TEST split
Found 3742 videos in folder
Processing test split (3742 videos)...
Looking for videos in: data\AUTSL\test


 23%|██▎       | 847/3742 [00:29<02:25, 19.94it/s]

Error opening video: data\AUTSL\test\signer6_sample185_color.mp4


100%|██████████| 3742/3742 [02:21<00:00, 26.43it/s]



--- AUTSL test Summary ---
Total in CSV: 3742
Found: 3742
Not found: 0
 Saved 3741 videos to data\processed\autsl_test.pkl
 Frames shape: (3741, 8, 64, 64, 3)
 Labels shape: (3741,)
Successfully processed test split


In [7]:

def load_wlasl_metadata():
    """Load WLASL JSON metadata"""
    import json
    
    json_path = WLASL_DIR / "WLASL_v0.3.json"
    
    with open(json_path, 'r') as f:
        wlasl_data = json.load(f)
    
    # Create mapping from video_id to label
    video_to_label = {}
    video_to_split = {}
    video_to_gloss = {}
    video_to_instances = {}
    
    for sign in wlasl_data:
        gloss = sign['gloss']
        instances = sign.get('instances', [])
        
        for inst in instances:
            video_id = inst.get('video_id')
            split = inst.get('split', 'unknown')
            
            if video_id:
                video_to_label[video_id] = sign.get('gloss_index', 0)
                video_to_split[video_id] = split
                video_to_gloss[video_id] = gloss
                video_to_instances[video_id] = inst
    
    return {
        'video_to_label': video_to_label,
        'video_to_split': video_to_split,
        'video_to_gloss': video_to_gloss,
        'video_to_instances': video_to_instances
    }


In [8]:
def preprocess_wlasl_dataset(split='train', num_frames=8, target_size=(64, 64)):
    """
    Preprocess WLASL dataset for a specific split
    FIXED: Correct path to videos folder with multiple extensions
    """
    
    # Load metadata
    metadata = load_wlasl_metadata()
    video_to_label = metadata['video_to_label']
    video_to_split = metadata['video_to_split']
    
    # Get videos for this split
    videos = [vid for vid, s in video_to_split.items() if s == split]
    
    # Load missing videos list
    missing_path = WLASL_DIR / "missing.txt"
    missing_ids = set()
    if missing_path.exists():
        with open(missing_path, 'r') as f:
            missing_ids = set([line.strip() for line in f.readlines()])
    
    # Filter out missing videos
    videos = [vid for vid in videos if vid not in missing_ids]
    
    print(f"Processing {split} split ({len(videos)} videos)...")
    print(f"Looking for videos in: {WLASL_DIR / 'videos'}")
    
    data = {'frames': [], 'labels': [], 'video_ids': [], 'glosses': []}
    
    for video_id in tqdm(videos):
        # FIXED: Look in the videos folder with multiple extensions
        video_path = None
        video_folder = WLASL_DIR / "videos"
        
        if video_folder.exists():
            for ext in ['.mp4', '.mkv', '.webm', '.avi', '.mov']:
                candidate = video_folder / f"{video_id}{ext}"
                if candidate.exists():
                    video_path = candidate
                    break
        
        if video_path is None:
            # Don't print every warning - too many
            continue
        
        # Extract frames
        frames = extract_frames_from_video(video_path, num_frames, target_size)
        
        if frames is not None:
            data['frames'].append(frames)
            data['labels'].append(video_to_label.get(video_id, -1))
            data['video_ids'].append(video_id)
            data['glosses'].append(metadata['video_to_gloss'].get(video_id, ''))
    
    if len(data['frames']) == 0:
        print(f"ERROR: No videos found for {split} split!")
        print(f"Checked in: {WLASL_DIR / 'videos'}")
        print(f"Expected video IDs like: {videos[:5] if videos else 'none'}")
        return None
    
    # Convert to numpy arrays
    data['frames'] = np.array(data['frames'], dtype=np.uint8)  # Use uint8 to save memory
    data['labels'] = np.array(data['labels'])
    
    # Save to disk
    save_path = PROCESSED_DIR / f"wlasl_{split}.pkl"
    with open(save_path, 'wb') as f:
        pickle.dump(data, f)
    
    print(f"Saved {len(data['frames'])} videos to {save_path}")
    print(f"  Frames shape: {data['frames'].shape}")
    print(f"  Memory: {data['frames'].nbytes / 1024**3:.2f} GB")
    
    return data

In [9]:
def process_all_wlasl():
    """Process all WLASL splits with error handling"""
    splits = ['train', 'val', 'test']
    
    # First, check if videos exist
    video_folder = WLASL_DIR / "videos"
    if not video_folder.exists():
        print(f"ERROR: Video folder not found at {video_folder}")
        print("Make sure your WLASL videos are in data/WLASL/videos/")
        return
    
    video_count = len(list(video_folder.glob("*")))
    print(f"Found {video_count} files in {video_folder}")
    
    for split in splits:
        print(f"\n--- Processing WLASL {split} ---")
        result = preprocess_wlasl_dataset(split=split, num_frames=8, target_size=(64, 64))
        if result is None:
            print(f"Failed to process {split} split. Check video paths.")

In [10]:
process_all_wlasl()

Found 11980 files in data\WLASL\videos

--- Processing WLASL train ---
Processing train split (8313 videos)...
Looking for videos in: data\WLASL\videos


100%|██████████| 8313/8313 [17:11<00:00,  8.06it/s]


Saved 8313 videos to data\processed\wlasl_train.pkl
  Frames shape: (8313, 8, 64, 64, 3)
  Memory: 0.76 GB

--- Processing WLASL val ---
Processing val split (2253 videos)...
Looking for videos in: data\WLASL\videos


100%|██████████| 2253/2253 [04:23<00:00,  8.56it/s]


Saved 2253 videos to data\processed\wlasl_val.pkl
  Frames shape: (2253, 8, 64, 64, 3)
  Memory: 0.21 GB

--- Processing WLASL test ---
Processing test split (1414 videos)...
Looking for videos in: data\WLASL\videos


100%|██████████| 1414/1414 [02:58<00:00,  7.92it/s]


Saved 1414 videos to data\processed\wlasl_test.pkl
  Frames shape: (1414, 8, 64, 64, 3)
  Memory: 0.13 GB


In [11]:
def preprocess_emosign_dataset():
    """
    Preprocess EMOSIGN dataset - LABELS ONLY
    EMOSIGN videos require ASLLRP access, so we only store labels
    """
    csv_path = EMOSIGN_DIR / "emosign_dataset.csv"
    
    if not csv_path.exists():
        print(f"WARNING: EMOSIGN CSV not found at {csv_path}")
        return None, []
    
    df = pd.read_csv(csv_path)
    
    # Define emotion columns (based on your CSV)
    emotion_cols = ['joy', 'excited', 'surprise_pos', 'surprise_neg', 'worry', 
                    'sadness', 'fear', 'disgust', 'frustration', 'anger']
    
    # Filter to only existing columns
    existing_emotion_cols = [col for col in emotion_cols if col in df.columns]
    
    print(f"Processing EMOSIGN ({len(df)} samples) - Labels only")
    print(f"Emotion columns found: {existing_emotion_cols}")
    
    data = {
        'video_names': df['video_name'].values,
        'sentiment_scores': df['Sentiment'].values.astype(np.float32),
        'emotion_intensities': df[existing_emotion_cols].values.astype(np.float32),
        'reasoning_1': df['Reasoning_1'].values,
        'reasoning_2': df['Reasoning_2'].values,
        'reasoning_3': df['Reasoning_3'].values,
    }
    
    # Save to disk
    save_path = PROCESSED_DIR / "emosign_labels.pkl"
    with open(save_path, 'wb') as f:
        pickle.dump(data, f)
    
    print(f"Saved {len(data['sentiment_scores'])} label samples to {save_path}")
    
    return data, existing_emotion_cols

In [12]:
emosign_data, emotion_columns = preprocess_emosign_dataset()

Processing EMOSIGN (200 samples) - Labels only
Emotion columns found: ['joy', 'excited', 'surprise_pos', 'surprise_neg', 'worry', 'sadness', 'fear', 'disgust', 'frustration', 'anger']
Saved 200 label samples to data\processed\emosign_labels.pkl


In [13]:

class AUTSLDataset(Dataset):
    """AUTSL Dataset for Multi-Task Learning"""
    
    def __init__(self, split='train', transform=None, augment=False):
        """
        Args:
            split: 'train', 'val', or 'test'
            transform: Optional transforms to apply
            augment: Whether to apply data augmentation
        """
        # Load preprocessed data
        data_path = PROCESSED_DIR / f"autsl_{split}.pkl"
        with open(data_path, 'rb') as f:
            self.data = pickle.load(f)
        
        self.frames = torch.FloatTensor(self.data['frames'])  # [N, T, H, W, 3]
        self.labels = torch.LongTensor(self.data['labels'])
        
        # Normalize frames to [0, 1]
        self.frames = self.frames / 255.0
        
        # Reshape to [N, T, C, H, W] for PyTorch
        self.frames = self.frames.permute(0, 1, 4, 2, 3)
        
        self.transform = transform
        self.augment = augment
        
    def __len__(self):
        return len(self.frames)
    
    def __getitem__(self, idx):
        frames = self.frames[idx]
        label = self.labels[idx]
        
        # Data augmentation (only during training)
        if self.augment:
            frames = self._augment_frames(frames)
        
        if self.transform:
            frames = self.transform(frames)
        
        return frames, label
    
    def _augment_frames(self, frames):
        
        # Random brightness adjustment
        if torch.rand(1) > 0.5:
            brightness = 0.8 + 0.4 * torch.rand(1)
            frames = frames * brightness
        
        return frames


class WLASLDataset(Dataset):
    """WLASL Dataset for Multi-Task Learning (with class balancing)"""
    
    def __init__(self, split='train', transform=None, balance_classes=True):
        data_path = PROCESSED_DIR / f"wlasl_{split}.pkl"
        with open(data_path, 'rb') as f:
            self.data = pickle.load(f)
        
        self.frames = torch.FloatTensor(self.data['frames'])
        self.labels = torch.LongTensor(self.data['labels'])
        
        # Normalize
        self.frames = self.frames / 255.0
        self.frames = self.frames.permute(0, 1, 4, 2, 3)

        
        self.transform = transform
        self.sampler = None 
        
        # Class balancing for training
        if balance_classes and split == 'train':
            self._balance_classes()
        

    
    def _balance_classes(self):
        """Balance dataset by oversampling minority classes"""
        from torch.utils.data import WeightedRandomSampler
        
        class_counts = torch.bincount(self.labels)
        class_weights = 1.0 / (class_counts.float() + 1e-6)
        sample_weights = class_weights[self.labels]
        
        self.sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
    
    def __len__(self):
        return len(self.frames)
    
    def __getitem__(self, idx):
        frames = self.frames[idx]
        label = self.labels[idx]
        
        if self.transform:
            frames = self.transform(frames)
        
        return frames, label


class EMOSIGNDataset(Dataset):
    """EMOSIGN Dataset for Emotion Recognition - LABELS ONLY"""
    
    def __init__(self, transform=None):
        data_path = PROCESSED_DIR / "emosign_labels.pkl"
        
        if not data_path.exists():
            print(f"ERROR: EMOSIGN labels not found at {data_path}")
            print("Run preprocess_emosign_dataset() first")
            self.frames = torch.zeros(1)
            self.sentiment = torch.zeros(1)
            self.emotions = torch.zeros((1, 10))
            return
        
        with open(data_path, 'rb') as f:
            self.data = pickle.load(f)
        
        # Note: No frames for EMOSIGN (labels only)
        self.sentiment = torch.FloatTensor(self.data['sentiment_scores'])
        self.emotions = torch.FloatTensor(self.data['emotion_intensities'])
        
        # Normalize sentiment from -3..3 to 0..1
        self.sentiment = (self.sentiment + 3) / 6
        
        # Normalize emotions from 1..4 to 0..1
        self.emotions = (self.emotions - 1) / 3
        
        self.transform = transform
    
    def __len__(self):
        return len(self.sentiment)
    
    def __getitem__(self, idx):
        sentiment = self.sentiment[idx]
        emotions = self.emotions[idx]
        
        # Return dummy frames if needed for compatibility with training loop
        # Actual emotion model would use different architecture
        dummy_frames = torch.zeros(16, 3, 112, 112)
        
        if self.transform:
            dummy_frames = self.transform(dummy_frames)
        
        return dummy_frames, sentiment, emotions

In [14]:
from torchvision import transforms


In [15]:
def create_data_loaders(batch_size=16, num_workers=4):
    
    # Define transforms
    train_transform = transforms.Compose([
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    # AUTSL loaders
    autsl_train = AUTSLDataset(split='train', transform=train_transform, augment=True)
    autsl_val = AUTSLDataset(split='val', transform=val_transform)
    autsl_test = AUTSLDataset(split='test', transform=val_transform)
    
    autsl_train_loader = DataLoader(
        autsl_train, batch_size=batch_size, shuffle=True, 
        num_workers=num_workers, pin_memory=True
    )
    autsl_val_loader = DataLoader(
        autsl_val, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )
    autsl_test_loader = DataLoader(
        autsl_test, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )
    
    # WLASL loaders
    wlasl_train = WLASLDataset(split='train', transform=train_transform, balance_classes=True)
    wlasl_val = WLASLDataset(split='val', transform=val_transform)
    wlasl_test = WLASLDataset(split='test', transform=val_transform)
    
    wlasl_train_loader = DataLoader(
        wlasl_train, batch_size=batch_size, 
        sampler=getattr(wlasl_train, 'sampler', None),
        shuffle=wlasl_train.sampler is None,
        num_workers=num_workers, pin_memory=True
    )
    wlasl_val_loader = DataLoader(
        wlasl_val, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )
    wlasl_test_loader = DataLoader(
        wlasl_test, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )
    
    # EMOSIGN loader
    emosign_path = PROCESSED_DIR / "emosign_labels.pkl"
    if emosign_path.exists():
        emosign_dataset = EMOSIGNDataset(transform=val_transform)
        emosign_loader = DataLoader(
            emosign_dataset, batch_size=batch_size, shuffle=True,
            num_workers=num_workers, pin_memory=True
        )
    else:
        print("EMOSIGN labels not found. Run preprocess_emosign_dataset() first.")
        emosign_loader = None
    
    loaders = {
        'autsl': {'train': autsl_train_loader, 'val': autsl_val_loader, 'test': autsl_test_loader},
        'wlasl': {'train': wlasl_train_loader, 'val': wlasl_val_loader, 'test': wlasl_test_loader},
        'emosign': emosign_loader
    }
    
    print("Data loaders created successfully!")
    print(f"AUTSL - Train: {len(autsl_train)}, Val: {len(autsl_val)}, Test: {len(autsl_test)}")
    print(f"WLASL - Train: {len(wlasl_train)}, Val: {len(wlasl_val)}, Test: {len(wlasl_test)}")
    print(f"EMOSIGN - Total: {len(emosign_dataset)}")
    
    return loaders

In [16]:
loaders = create_data_loaders(batch_size=16)

Data loaders created successfully!
AUTSL - Train: 28132, Val: 4418, Test: 3741
WLASL - Train: 8313, Val: 2253, Test: 1414
EMOSIGN - Total: 200
